Чтение данных

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

In [ ]:
def reading(filename):
  result = []
  raw_data = []
  with open(filename, 'r', encoding='utf-16') as f:
    for line in f:
      raw_data.append(line)

  for line in raw_data[1:]:
      line = line.split("\t")
      if len(line[0]) > 0:
        elem = dict()
        elem['text'] = line[0]
        elem['sent_rel'] = line[1]
        elem['sent_type'] = line[2]
        elem['stance'] = line[3]
        elem['argument'] = line[4]
        elem['abstract_number'] = line[5]
        result.append(elem)
  return result

In [ ]:
bilingualism_data, relativity_data, ug_data = [], [], []
bilingualism_data = reading("/content/bilingualism.txt")
relativity_data = reading("/content/relativity.txt")
ug_data = reading("/content/universal grammar.txt")

def processing(data, num):
  abs_num = 1
  sent_num = 1
  for elem in data:
    prev = abs_num
    elem['abstract_rel'] = num
    if elem['abstract_number'] == prev:
      sent_num += 1
    else:
      sent_num = 1
    elem['sent_number'] = sent_num
    abs_num = elem['abstract_number']
  return data
bilingualism_data = processing(bilingualism_data, 0)
ug_data = processing(ug_data, 1)
relativity_data = processing(relativity_data, 2)

## Определение типа предложения (1 - introduction, 2 - method, 3 - results)

BiLSTM (без предобучения)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes,
                 embedding_matrix=None, dropout=0.5, freeze_embeddings=False):
        super(BiLSTMClassifier, self).__init__()

        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.FloatTensor(embedding_matrix),
                freeze=freeze_embeddings,
                padding_idx=0
            )
            self.embedding_dim = embedding_matrix.shape[1]
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
            self.embedding_dim = embedding_dim

        self.lstm = nn.LSTM(
            input_size=self.embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)  # *2 для bidirectional

    def forward(self, input_ids):
        # Получаем эмбеддинги
        embedded = self.embedding(input_ids)  # [batch_size, seq_len, embedding_dim]

        # Пропускаем через BiLSTM
        lstm_output, (hidden, cell) = self.lstm(embedded)

        # Берем последние скрытые состояния из BiLSTM (оба направления)
        # hidden shape: [num_layers * 2, batch_size, hidden_size]
        hidden_forward = hidden[-2, :, :]  # последний слой, прямое направление
        hidden_backward = hidden[-1, :, :]  # последний слой, обратное направление
        hidden_concat = torch.cat((hidden_forward, hidden_backward), dim=1)

        # Классификация
        hidden_concat = self.dropout(hidden_concat)
        logits = self.classifier(hidden_concat)

        return logits

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
def sentence_type_bilstm(data, tokenizer, dataset_type,
                         embedding_dim=300, hidden_size=256, num_layers=2,
                         batch_size=16, num_epochs=10, learning_rate=2e-5):
    # мэппинг меток
    label2id = {'1': 0, '2': 1, '3': 2}
    cache_filename = f"embeddings_{dataset_type}_bilstm.npz"

    print(f"Подготовка данных для {dataset_type}...")
    texts = [item['text'] for item in data]
    labels = [label2id[item['sent_type']] for item in data]

    dataset = TextDataset(texts, labels, tokenizer)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
    all_true_labels = []
    all_pred_labels = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используется устройство: {device}")

    # Кросс-валидация
    for fold, (train_idx, val_idx) in enumerate(cv.split(texts, labels), 1):
        print(f"\n=== Fold {fold}/5 ===")

        train_dataset = torch.utils.data.Subset(dataset, train_idx)
        val_dataset = torch.utils.data.Subset(dataset, val_idx)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        vocab_size = tokenizer.vocab_size
        num_classes = 3 

        model = BiLSTMClassifier(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            num_classes=num_classes,
            dropout=0.5
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(num_epochs):
            model.train()
            total_loss = 0

            for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
                input_ids = batch['input_ids'].to(device)
                labels_batch = batch['label'].to(device)

                optimizer.zero_grad()
                outputs = model(input_ids)
                loss = criterion(outputs, labels_batch)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)

            model.eval()
            val_preds = []
            val_labels = []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(device)
                    labels_batch = batch['label'].to(device)

                    outputs = model(input_ids)
                    preds = torch.argmax(outputs, dim=1)

                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(labels_batch.cpu().numpy())

            val_acc = accuracy_score(val_labels, val_preds)
            print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}, Val Accuracy = {val_acc:.4f}')
        all_true_labels.extend(val_labels)
        all_pred_labels.extend(val_preds)

    # Вывод результатов
    print("\n=== Результаты кросс-валидации ===")
    print(classification_report(all_true_labels, all_pred_labels,
                                target_names=['introduction', 'method', 'results']))
    print(f"Accuracy: {accuracy_score(all_true_labels, all_pred_labels):.3f}")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
sentence_type_bilstm(
    bilingualism_data,
    tokenizer,
    "biling_type",
    embedding_dim=300,     
    hidden_size=256,    
    num_layers=2,        
    batch_size=16,      
    num_epochs=10,         
    learning_rate=2e-5)      

In [ ]:
sentence_type_bilstm(
    relativity_data,
    tokenizer,
    "type",
    embedding_dim=300,     
    hidden_size=256,    
    num_layers=2,        
    batch_size=16,      
    num_epochs=10,         
    learning_rate=2e-5)      

In [ ]:
sentence_type_bilstm(
    ug_data,
    tokenizer,
    "ug_type",
    embedding_dim=300,     
    hidden_size=256,    
    num_layers=2,        
    batch_size=16,      
    num_epochs=10,         
    learning_rate=2e-5)   

Предобученная BiLSTM (из Lauscher et al. 2018)

In [ ]:
#Загрузка модели
!wget -r -np -nH --cut-dirs=2 -R "index.html*" http://data.dws.informatik.uni-mannheim.de/arguminsci/model/

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import re

In [ ]:
tf.compat.v1.disable_eager_execution()
class ArguminSciFeatureExtractor:
    
    def __init__(self, model_type=None, model_root='/content'):
        self.model_path = os.path.join(model_root, model_type) if model_type else model_root
        
        config_path = os.path.join(self.model_path, 'config.txt')
        if os.path.exists(config_path):
            with open(config_path, 'r') as f:
                self.config = f.read()
            import ast
            try:
                config_lines = self.config.strip().split('\n')
                for line in config_lines:
                    if 'main labels:' in line:
                        self.labels = ast.literal_eval(line.split('main labels:')[1].strip())
                        print(f"Найдены метки: {self.labels}")
                    if 'batch size:' in line:
                        self.batch_size = int(line.split(':')[1].strip())
            except:
                self.labels = None

        checkpoint_prefix = os.path.join(self.model_path, 'best-model')
        
        self.graph = tf.Graph()
        self.sess = tf.compat.v1.Session(graph=self.graph)
        
        with self.graph.as_default():
            try:
                saver = tf.compat.v1.train.import_meta_graph(f'{checkpoint_prefix}.meta')
                saver.restore(self.sess, checkpoint_prefix)
                
                ops = self.graph.get_operations()
                self.input_tensor = None
                possible_input_names = ['input_x:0', 'inputs:0', 'input:0', 'Placeholder:0']
                for name in possible_input_names:
                    try:
                        self.input_tensor = self.graph.get_tensor_by_name(name)

                        break
                    except:
                        continue
                if self.input_tensor is None:
                    for op in ops:
                        if op.type == 'Placeholder':
                            self.input_tensor = op.outputs[0]
                            break
                
                self.feature_tensor = None                
                if self.feature_tensor is None:
                    for op in reversed(ops):
                        if len(op.outputs) > 0:
                            try:
                                output_shape = op.outputs[0].shape
                                if output_shape.rank is not None:
                                    if len(output_shape) == 2 and output_shape[1] == 6:
                                        self.feature_tensor = op.outputs[0]
                                        break
                            except:
                                continue
                
                self.input_dim = self.input_tensor.shape[1]
                self.feature_dim = self.feature_tensor.shape[1] if self.feature_tensor.shape[1] is not None else 6
            except Exception as e:
                if self.verbose:
                    print(f"Ошибка загрузки модели: {e}")
                raise
                
        self._load_embeddings_and_vocab()
    
    def _load_embeddings_and_vocab(self):
        embeddings_path = os.path.join(self.model_path, 'embeddings')
        self.embeddings = None
        
        if os.path.exists(embeddings_path):
            try:
                self.embeddings = np.load(embeddings_path, allow_pickle=True)
            except:
                with open(embeddings_path, 'rb') as f:
                    self.embeddings = pickle.load(f)
        embedding_vocab_path = os.path.join(self.model_path, 'embedding_vocab')
        self.vocab = None
        
        if os.path.exists(embedding_vocab_path):
            try:
                with open(embedding_vocab_path, 'rb') as f:
                    self.vocab = pickle.load(f)
            except:
                    self.vocab = np.load(embedding_vocab_path, allow_pickle=True).item()
        
        if self.vocab is None and self.embeddings is not None:
            self.vocab = {f"word_{i}": i for i in range(min(10000, len(self.embeddings)))}
    
    def text_to_features(self, text):
        tokens = re.findall(r'\w+|[^\w\s]', text.lower())
        token_ids = []
        for token in tokens:
            if token in self.vocab:
                token_ids.append(self.vocab[token])
            else:
                token_ids.append(0) 
        if self.embeddings is not None:
            max_tokens = self.input_dim // self.embeddings.shape[1]
            token_embeddings = []
            for token_id in token_ids[:max_tokens]:
                if token_id < len(self.embeddings):
                    token_embeddings.append(self.embeddings[token_id])
                else:
                    token_embeddings.append(np.zeros(self.embeddings.shape[1]))
            if token_embeddings:
                features = np.concatenate(token_embeddings)
            else:
                features = np.zeros(self.input_dim)
            if len(features) > self.input_dim:
                features = features[:self.input_dim]
            elif len(features) < self.input_dim:
                features = np.pad(features, (0, self.input_dim - len(features)), 'constant')
        else:
            features = np.zeros(self.input_dim)
            for token_id in token_ids[:self.input_dim]:
                if token_id < self.input_dim:
                    features[token_id] += 1
        
        return features.reshape(1, -1).astype(np.float32)
    
    def extract_features(self, texts, batch_size=32):
        features_list = []
        
        for i, text in enumerate(texts):
            input_features = self.text_to_features(text)
            try:
                with self.graph.as_default():
                    feed_dict = {self.input_tensor: input_features}
                    features = self.sess.run(self.feature_tensor, feed_dict=feed_dict)
                    if len(features.shape) > 1:
                        features = features.flatten()
                    
                    features_list.append(features)
            except Exception as e:
                features_list.append(np.zeros(self.feature_dim))
        
        return np.array(features_list)

In [ ]:
def prepare_data_and_train_classifier(data, model_type='discourse', test_size=0.2, classifier_type='rf'):
    texts = [item['text'] for item in data]
    labels = [str(item['sent_type']) for item in data] 
    unique_labels = sorted(set(labels))
    label2id = {label: idx for idx, label in enumerate(unique_labels)}
    id2label = {idx: label for label, idx in label2id.items()} 
    y = np.array([label2id[label] for label in labels])
    for label in unique_labels:
        count = labels.count(label)
        print(f"  {label}: {count} ({count/len(labels)*100:.1f}%)")
    
    extractor = ArguminSciFeatureExtractor(model_type=model_type, model_root='/content')
    X = extractor.extract_features(texts)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    if classifier_type == 'rf':
        classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        classifier_name = "Случайный лес"
    
    classifier.fit(X_train_scaled, y_train)
    y_pred = classifier.predict(X_test_scaled)
    
    print(f"\n{'='*50}")
    print(f"РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ ({classifier_name})")
    print(f"{'='*50}")
    
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=unique_labels))
    
    model_dir = '/content/trained_classifier'
    os.makedirs(model_dir, exist_ok=True)
    
    with open(os.path.join(model_dir, 'classifier.pkl'), 'wb') as f:
        pickle.dump(classifier, f)
    with open(os.path.join(model_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    with open(os.path.join(model_dir, 'label_encoder.pkl'), 'wb') as f:
        pickle.dump(label2id, f)
    
    return {
        'classifier': classifier,
        'scaler': scaler,
        'label2id': label2id,
        'id2label': id2label,
        'extractor': extractor,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'y_pred': y_pred
    }

In [ ]:
results = prepare_data_and_train_classifier(
        data=bilingualism_data,
        model_type='discourse',
        test_size=0.2,
        classifier_type='rf' 
    )


In [ ]:
results = prepare_data_and_train_classifier(
        data=relativity_data,
        model_type='discourse',
        test_size=0.2,
        classifier_type='rf' 
    )


In [ ]:
results = prepare_data_and_train_classifier(
        data=ug_data,
        model_type='discourse',
        test_size=0.2,
        classifier_type='rf' 
    )


RoBERTa large

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
roberta_model = AutoModel.from_pretrained("roberta-large")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_model.to(device)
roberta_model.eval()

In [ ]:
def sentence_type(data, model, tokenizer, classifier, dataset_type):
    # мэппинг меток
    label2id = {'1': 0, '2': 1, '3': 2}
    cache_filename = f"embeddings_{dataset_type}_{model.__class__.__name__}.npy"

    # эмбеддинги
    def get_cls_embedding(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        return cls_embedding

    # проверяем, есть ли сохраненные эмбеддинги
    try:
        # пробуем загрузить сохраненные эмбеддинги
        saved_data = np.load(cache_filename, allow_pickle=True)
        X = saved_data['X']
        y = saved_data['y']
        print(f"Загружены сохраненные эмбеддинги из {cache_filename}")
    except (FileNotFoundError, IOError):
        # если файла нет, вычисляем эмбеддинги заново
        print(f"Вычисляются новые эмбеддинги для {dataset_type}...")
        X = []
        y = []

        for item in data:
            emb = get_cls_embedding(item['text'])
            X.append(emb)
            y.append(label2id[item['sent_type']])

        X = np.array(X)
        y = np.array(y)

        # сохраняем эмбеддинги
        np.savez(cache_filename, X=X, y=y)
        print(f"Эмбеддинги сохранены в {cache_filename}")

    if classifier == "LogisticRegression":
      clf = LogisticRegression(max_iter=1000)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
    # кросс-валидация
    y_pred = cross_val_predict(clf, X, y, cv=cv)

    print(classification_report(y, y_pred, target_names=['introduction', 'method', 'results']))
    print(f"Accuracy: {accuracy_score(y, y_pred):.3f}")

In [ ]:
sentence_type(bilingualism_data, roberta_model, roberta_tokenizer, "LogisticRegression", "biling_type")

In [ ]:
sentence_type(ug_data, roberta_model, roberta_tokenizer, "LogisticRegression", "ug_type")

In [ ]:
sentence_type(relativity_data, roberta_model, roberta_tokenizer, "LogisticRegression", "relativity_type")

SciBERT

In [ ]:
model_name = "allenai/scibert_scivocab_cased"
scibert_tokenizer = AutoTokenizer.from_pretrained(model_name)
scibert_model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scibert_model.to(device)
scibert_model.eval()

In [ ]:
sentence_type(bilingualism_data, scibert_model, scibert_tokenizer, "LogisticRegression", "biling_type")

In [ ]:
sentence_type(ug_data, scibert_model, scibert_tokenizer, "LogisticRegression", "ug_type")

In [ ]:
sentence_type(relativity_data, scibert_model, scibert_tokenizer, "LogisticRegression", "relativity_type")

Добавление порядкового номера предложения (в аннотации)

In [ ]:
def sentence_type_context(data, model, tokenizer, classifier, dataset_type):
    # мэппинг меток
    label2id = {'1': 0, '2': 1, '3': 2}

    # создаем имя файла
    cache_filename = f"context_embeddings_{dataset_type}_{model.__class__.__name__}.npz"

    # эмбеддинги
    def get_cls_embedding(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        return cls_embedding

    # проверяем, есть ли сохраненные эмбеддинги
    try:
        # пробуем загрузить сохраненные эмбеддинги
        saved_data = np.load(cache_filename, allow_pickle=True)
        X = saved_data['X']
        y = saved_data['y']
        sent_numbers = saved_data['sent_numbers']
        print(f"Загружены сохраненные эмбеддинги из {cache_filename}")
    except (FileNotFoundError, IOError):
        # если файла нет, вычисляем эмбеддинги заново
        print(f"Вычисляются новые эмбеддинги для {dataset_type}...")
        X = []
        y = []
        sent_numbers = []

        for item in data:
            emb = get_cls_embedding(item['text'])
            X.append(emb)
            y.append(label2id[item['sent_type']])
            sent_numbers.append(item.get('sent_number', 0))

        X = np.array(X)
        y = np.array(y)
        sent_numbers = np.array(sent_numbers).reshape(-1, 1)

        # сохраняем эмбеддинги
        np.savez(cache_filename, X=X, y=y, sent_numbers=sent_numbers)
        print(f"Эмбеддинги сохранены в {cache_filename}")

    # объединяем эмбеддинги с номерами предложений
    X_combined = np.hstack([X, sent_numbers])

    if classifier == "LogisticRegression":
        clf = LogisticRegression(max_iter=1000)
    elif classifier == "LinearSVC":
        clf = LinearSVC(max_iter=5000)
    elif classifier == "GaussianNB":
        clf = GaussianNB()
    elif classifier == "RandomForest":
        clf = RandomForestClassifier(n_estimators=100, random_state=13)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
    # кросс-валидация
    y_pred = cross_val_predict(clf, X_combined, y, cv=cv)

    print(classification_report(y, y_pred, target_names=['introduction', 'method', 'results']))
    print(f"Accuracy: {accuracy_score(y, y_pred):.3f}")

In [ ]:
sentence_type_context(bilingualism_data, scibert_model, scibert_tokenizer, "LogisticRegression", "biling_type_context")

In [ ]:
sentence_type_context(ug_data, scibert_model, scibert_tokenizer, "LogisticRegression", "ug_type_context")

In [ ]:
sentence_type_context(relativity_data, scibert_model, scibert_tokenizer, "LogisticRegression", "relativity_type_context")

Тестирование других классификаторов с лучшими вариантами моделей (SciBERT с контекстом для типа предложения)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

In [ ]:
#LinearSVC
sentence_type_context(bilingualism_data, scibert_model, scibert_tokenizer, "LinearSVC", "biling_type_context")
sentence_type_context(ug_data, scibert_model, scibert_tokenizer, "LinearSVC", "ug_type_context")
sentence_type_context(relativity_data, scibert_model, scibert_tokenizer, "LinearSVC", "relativity_type_context")

In [ ]:
#GaussianNB
sentence_type_context(bilingualism_data, scibert_model, scibert_tokenizer, "GaussianNB", "biling_type_context")
sentence_type_context(ug_data, scibert_model, scibert_tokenizer, "GaussianNB", "ug_type_context")
sentence_type_context(relativity_data, scibert_model, scibert_tokenizer, "GaussianNB", "relativity_type_context")

In [ ]:
#RandomForestClassifier
sentence_type_context(bilingualism_data, scibert_model, scibert_tokenizer, "RandomForest", "biling_type_context")
sentence_type_context(ug_data, scibert_model, scibert_tokenizer, "RandomForest", "ug_type_context")
sentence_type_context(relativity_data, scibert_model, scibert_tokenizer, "RandomForest", "relativity_type_context")

LLM

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm
import json
import re

def load_and_prepare_data(excel_file_path, sheet_name):
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    data = []
    for idx, row in df.iterrows():
        if pd.notna(row.get('text', '')):
            item = {
                'text': row['text'],
                'relevance': row.get('sent_rel', -1),
                'type': row.get('type', -1),
                'stance': row.get('stance', -1),
                'argument': row.get('arg', -1),
                'abstract_number': row.get('abstract_number', -1)
            }
            data.append(item)

    print (data[0])
    return data

def create_prompts_for_annotation(sentences, theme, task_type):
    """
    Создание промптов для различных задач аннотации.

    task_type: 'relevance', 'type', 'stance', 'argument'
    """
    prompts = []

    if task_type == 'relevance' and theme == "bilingualism":
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine whether it is relevant to the topic of bilingualism and cognitive advantages/disadvantages.
Answer ONLY with one of these labels:
- 1: the sentence is relevant (discusses bilingual advantage/disadvantage, executive functions, cognitive reserve, dementia, etc.)
- -1: the sentence is not relevant (general statements, background, methods description, etc.)

Sentence: {sentence}
Label: """

    elif task_type == 'relevance' and theme == "universal grammar":
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine whether it is relevant to the topic of universal grammar and innate language faculty.
Answer ONLY with one of these labels:
- 1: the sentence is relevant (discusses universal grammar, innateness of language faculty, learnability etc.)
- -1: the sentence is not relevant (general statements, background, methods description, etc.)

Sentence: {sentence}
Label: """
    elif task_type == 'relevance' and theme == "linguistic relativity ":
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine whether it is relevant to the topic of linguistic relativity, the correlation between language and thinking.
Answer ONLY with one of these labels:
- 1: the sentence is relevant (discusses bilingual advantage/disadvantage, executive functions, cognitive reserve, dementia, etc.)
- -1: the sentence is not relevant (general statements, background, methods description, etc.)

Sentence: {sentence}
Label: """
    elif task_type == 'type':
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, classify its structural type:
- 1: Literature review (known facts, background, previous research)
- 2: Method (description of study design, participants, procedures, description of the presentation sequence)
- 3: Results/Position (new findings, author's contribution, conclusions)
Answer ONLY with the number (1, 2, or 3).

Sentence: {sentence}
Label: """

    elif task_type == 'stance'and theme == "bilingualism":
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine the author's stance on the bilingual advantage hypothesis:
- 2: For (supports the bilingual advantage, implies that bilinguals perform cognitively better, than monolinguals)
- 1: Mixed/Other (ambiguous, contradictory, no clear position)
- 0: Against (opposes or questions the bilingual advantage)
- -1: No position (descriptive statements without evaluation)
Answer ONLY with the number (2, 1, 0, or -1).

Sentence: {sentence}
Label: """

    elif task_type == 'stance' and theme == "linguistic relativity ":
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine the author's stance on the linguistic relativity hypothesic:
- 2: For (supports linguistic relativity hypothesis, implies that language determines thinking)
- 1: Mixed/Other (ambiguous, contradictory, no clear position)
- 0: Against (opposes or questions the linguistic relativity hypothesis, implies that language does not determine thinking)
- -1: No position (descriptive statements without evaluation)
Answer ONLY with the number (2, 1, 0, or -1).

Sentence: {sentence}
Label: """

    elif task_type == 'stance' and theme == "universal grammar":
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine the author's stance on the theory of universal grammar:
- 2: For (supports the theory of universal grammar, implies that language faculty is innate and universal)
- 1: Mixed/Other (ambiguous, contradictory, no clear position)
- 0: Against (opposes or questions the universal grammar theory, implies that language faculty is not innate and universal)
- -1: No position (descriptive statements without evaluation)
Answer ONLY with the number (2, 1, 0, or -1).

Sentence: {sentence}
Label: """

    elif task_type == 'argument':
        instruction = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine whether it contains an argument:
- 1: Contains an argument (provides reasoning, evidence, or justification)
- -1: Does not contain an argument (just states a fact or position without support)
Answer ONLY with the number (1 or -1).

Sentence: {sentence}
Label: """

    else:
        raise ValueError(f"Unknown task_type: {task_type}")

    for sentence in sentences:
        prompt = instruction.format(sentence=sentence)
        prompts.append(prompt)

    return prompts


In [ ]:
class LinguisticAnnotationPredictor:
    def __init__(self, model_name, quantization_config=None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.padding_side = "left"
        self.tokenizer.pad_token = self.tokenizer.eos_token

        if quantization_config:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=quantization_config,
                device_map="auto"
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

        self.model.eval()

    def predict_batch(self, prompts, batch_size=8, max_new_tokens=10):
        predictions = []

        for i in tqdm(range(0, len(prompts), batch_size), desc="Predicting"):
            batch_prompts = prompts[i:i+batch_size]

            messages_batch = []
            for prompt in batch_prompts:
                messages = [
                    {"role": "system", "content": "You are a linguistics researcher. Answer concisely and only with the required label."},
                    {"role": "user", "content": prompt}
                ]
                messages_batch.append(messages)

            inputs = []
            for messages in messages_batch:
                text = self.tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False
                )
                inputs.append(text)

            encodings = self.tokenizer(
                inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to("cuda")

            with torch.no_grad():
                outputs = self.model.generate(
                    **encodings,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )


            for output in outputs:
                generated = self.tokenizer.decode(output, skip_special_tokens=True)

                if "<|im_start|>assistant" in generated:
                    answer = generated.split("<|im_start|>assistant")[-1].strip()
                else:
                    answer = generated.split("\n")[-1].strip()


                match = re.search(r'[-]?\d+', answer)
                if match:
                    pred = int(match.group())
                else:
                    pred = -1

                predictions.append(pred)

        return predictions


In [ ]:
def evaluate_predictions(y_true, y_pred, task_name):

    valid_mask = [y != -1 for y in y_true]
    y_true_filtered = [y for y, m in zip(y_true, valid_mask) if m]
    y_pred_filtered = [y for y, m in zip(y_pred, valid_mask) if m]

    if len(y_true_filtered) == 0:
        print(f"Warning: No valid samples for {task_name}")
        return

    print(f"\n{'='*50}")
    print(f"Evaluation for {task_name}")
    print(f"{'='*50}")
    print(f"Number of samples: {len(y_true_filtered)}")
    print(f"Accuracy: {accuracy_score(y_true_filtered, y_pred_filtered):.4f}")
    print(f"Macro F1: {f1_score(y_true_filtered, y_pred_filtered, average='macro'):.4f}")
    print(f"Weighted F1: {f1_score(y_true_filtered, y_pred_filtered, average='weighted'):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true_filtered, y_pred_filtered, zero_division=0))
    
def main(SHEET_NAME):
    EXCEL_FILE = "разметка пзкл.xlsx"
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

    print("Loading data...")
    data = load_and_prepare_data(EXCEL_FILE, SHEET_NAME)
    print(f"Loaded {len(data)} sentences")

    sentences = [item['text'] for item in data]
    true_labels = {
        'relevance': [item['relevance'] for item in data],
        'type': [item['type'] for item in data],
        'stance': [item['stance'] for item in data],
        'argument': [item['argument'] for item in data]
    }


    print("\nLoading model...")
    predictor = LinguisticAnnotationPredictor(MODEL_NAME)


    results = {}

    for task in ['stance', 'type']:

        print(f"\n{'='*50}")
        print(f"Processing {task} classification...")
        print(f"{'='*50}")


        prompts = create_prompts_for_annotation(sentences, SHEET_NAME, task)

        predictions = predictor.predict_batch(prompts, batch_size=8)

        evaluate_predictions(true_labels[task], predictions, task)

        results[task] = {
            'true': true_labels[task],
            'pred': predictions
        }

    with open('annotation_results.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("\nResults saved to annotation_results.json")


In [ ]:
def get_few_shot_examples_by_sheet(sheet_name, task_type):
    """
    Возвращает отформатированную строку с примерами для few-shot.
    """

    if sheet_name == "bilingualism" and task_type == "stance":
        examples_list = [
            {
                'sentence': "Firstly, bilinguals are claimed to acquire a third language easier than monolinguals acquire a second.",
                'label': 2
            },
            {
                'sentence': "Conclusions — This study does not support a protective effect of bilingualism on age-related cognitive decline or the development of dementia.",
                'label': 0
            },
            {
                'sentence': "We tested their performance in a verbal Stroop task and in a nonverbal version of the same task (the number size-congruency task).",
                'label': -1
            },
            {
                'sentence': "Some have suggested that bilinguals also have greater working memory capacity than comparable monolinguals, but evidence for this suggestion is mixed.",
                'label': 1
            }
        ]

    elif sheet_name == "linguistic relativity " and task_type == "stance":
        examples_list = [
            {
                'sentence': "This difference between the two languages is reflected in the way their speakers think about time.",
                'label': 2
            },
            {
                'sentence': "The idea that language shapes the way we think, often associated with Benjamin Whorf, has long been decried as not only wrong but also fundamentally wrong-headed.",
                'label': 0
            },
            {
                'sentence': "Participants sorted a representative set of 65 colours into ‘N’ groups (where N ranged from 2 through to 12) based on their perceptual similarity.",
                'label': -1
            },
            {
                'sentence': "The Sapir-Whorf hypothesis is reevaluated in the light of these results.",
                'label': 1
            }
        ]

    elif sheet_name == "universal grammar" and task_type == "stance":
        examples_list = [
            {
                'sentence': "We conclude that a biologically determined Universal Grammar is not evolutionarily viable.",
                'label': 0
            },
            {
                'sentence': "Consequently, such apparent exceptionality argues strongly for Universal Grammar and against E&L.",
                'label': 2
            },
            {
                'sentence': "This commentary aims to highlight what exactly is controversial about the traditional Universal Grammar (UG) hypothesis and what is not.",
                'label': 1
            },
            {
                'sentence': "ISN is one of the very few languages that linguists have had the chance to study as it was created and evolved.",
                'label': -1
            }
        ]

    elif sheet_name == "bilingualism" and task_type == "type":
        examples_list = [
            {
                'sentence': "The results indicated that individuals’ degree of bilingualism and creativity are positively correlated, regardless of gender or age.",
                'label': 3
            },
            {
                'sentence': "We tested their performance in a verbal Stroop task and in a nonverbal version of the same task (the number size-congruency task).",
                'label': 2
            },
            {
                'sentence': "Previous studies have identified bilingualism as a protective factor against dementia.",
                'label': 1
            }
        ]

    elif sheet_name == "linguistic relativity " and task_type == "type":
        examples_list = [
            {
                'sentence': "In short, the present research yielded no support for the Sapir-Whorf hypothesis.",
                'label': 3
            },
            {
                'sentence': "Participants sorted a representative set of 65 colours into ‘N’ groups (where N ranged from 2 through to 12) based on their perceptual similarity.",
                'label': 2
            },
            {
                'sentence': "The idea that language shapes the way we think, often associated with Benjamin Whorf, has long been decried as not only wrong but also fundamentally wrong-headed.",
                'label': 1
            }
        ]

    elif sheet_name == "universal grammar" and task_type == "type":
        examples_list = [
            {
                'sentence': "We conclude that a biologically determined Universal Grammar is not evolutionarily viable.",
                'label': 3
            },
            {
                'sentence': "This commentary aims to highlight what exactly is controversial about the traditional Universal Grammar (UG) hypothesis and what is not.",
                'label': 2
            },
            {
                'sentence': "Syntactic knowledge is widely held to be partially innate, rather than learned.",
                'label': 1
            }
        ]

    else:
        examples_list = [
            {
                'sentence': "Research has demonstrated that bilingualism confers cognitive advantages in executive functions.",
                'label': 2
            },
            {
                'sentence': "However, conflicting evidence has emerged questioning the robustness of the bilingual advantage.",
                'label': 0
            },
            {
                'sentence': "The present study examines the bilingual advantage in executive functions.",
                'label': -1
            }
        ]


    examples_string = ""
    for i, ex in enumerate(examples_list, 1):
        examples_string += f"Example {i}:\n"
        examples_string += f"Sentence: {ex['sentence']}\n"
        examples_string += f"Label: {ex['label']}\n\n"

    return examples_string


def create_combined_prompt(sentence, examples_string, task_type, theme):


    if task_type == 'type':
        instruction_part = """You are a linguistics researcher. For the given sentence from a scientific abstract, classify its structural type:
- 1: Literature review (known facts, background, previous research)
- 2: Method (description of study design, participants, procedures)
- 3: Results/Position (new findings, author's contribution, conclusions)

"""
    elif task_type == 'stance'and theme == "bilingualism":
      instruction_part = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine the author's stance on the bilingual advantage hypothesis:
- 2: For (supports the bilingual advantage, implies that bilinguals perform cognitively better, than monolinguals)
- 1: Mixed/Other (ambiguous, contradictory, no clear position)
- 0: Against (opposes or questions the bilingual advantage)
- -1: No position (descriptive statements without evaluation)
"""

    elif task_type == 'stance' and theme == "linguistic relativity ":
      instruction_part = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine the author's stance on the linguistic relativity hypothesic:
- 2: For (supports linguistic relativity hypothesis, implies that language determines thinking)
- 1: Mixed/Other (ambiguous, contradictory, no clear position)
- 0: Against (opposes or questions the linguistic relativity hypothesis, implies that language does not determine thinking)
- -1: No position (descriptive statements without evaluation)
"""

    elif task_type == 'stance' and theme == "universal grammar":
      instruction_part = """You are a linguistics researcher. For the given sentence from a scientific abstract, determine the author's stance on the theory of universal grammar:
- 2: For (supports the theory of universal grammar, implies that language faculty is innate and universal)
- 1: Mixed/Other (ambiguous, contradictory, no clear position)
- 0: Against (opposes or questions the universal grammar theory, implies that language faculty is not innate and universal)
- -1: No position (descriptive statements without evaluation)
"""
    else:
        instruction_part = f"""You are a linguistics researcher. Annotate the following sentences for {task_type}.

"""


    final_part = f"""Now annotate the following sentence. Answer ONLY with the label number.

Sentence: {sentence}
Label: """


    full_prompt = instruction_part + "\nHere are some examples:\n\n" + examples_string + final_part

    return full_prompt


def run_few_shot_evaluation(data, predictor, task_type, examples_string, theme, batch_size=8):

    sentences = [item['text'] for item in data]
    true_labels = [item[task_type] for item in data]


    prompts = []
    for sentence in sentences:
        combined_prompt = create_combined_prompt(sentence, examples_string, task_type, theme)
        prompts.append(combined_prompt)

    print("\n" + "="*60)
    print("Пример:")
    print("="*60)
    print(prompts[0][:1000] + "...")
    print("="*60)

    predictions = predictor.predict_batch(prompts, batch_size)

    evaluate_predictions(true_labels, predictions, f"{task_type}_fewshot")

    return predictions


def main_few_shot(SHEET_NAME, task_type):
    EXCEL_FILE = "разметка пзкл.xlsx"
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

    print(f"\n{'='*60}")
    print(f"FEW-SHOT EVALUATION")
    print(f"Sheet: {SHEET_NAME}")
    print(f"Task: {task_type}")
    print(f"{'='*60}")


    print("\nLoading data...")
    data = load_and_prepare_data(EXCEL_FILE, SHEET_NAME)
    print(f"Loaded {len(data)} sentences")


    examples_string = get_few_shot_examples_by_sheet(SHEET_NAME, task_type)

    print(f"\nExamples for few-shot:")
    print(examples_string)

    print("\nLoading model...")
    predictor = LinguisticAnnotationPredictor(MODEL_NAME)

    predictions = run_few_shot_evaluation(data, predictor, task_type, examples_string, SHEET_NAME)

    return predictions

LLM: zero-shot

In [ ]:
if __name__ == "__main__":
    main("universal grammar")

In [ ]:
if __name__ == "__main__":
    main("bilingualism")

In [ ]:
if __name__ == "__main__":
    main("linguistic relativity ")

LLM:few-shot

In [ ]:
if __name__ == "__main__":
    main_few_shot("bilingualism", task_type='type')

In [ ]:
if __name__ == "__main__":
    main_few_shot("universal grammar", task_type='type')

In [ ]:
if __name__ == "__main__":
    main_few_shot("linguistic relativity ", task_type='type')

## Определение позиции (за/против/другое/позиция отсутствует)

BiLSTM (без предобучения)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes,
                 embedding_matrix=None, dropout=0.5, freeze_embeddings=False):
        super(BiLSTMClassifier, self).__init__()

        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.FloatTensor(embedding_matrix),
                freeze=freeze_embeddings,
                padding_idx=0
            )
            self.embedding_dim = embedding_matrix.shape[1]
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
            self.embedding_dim = embedding_dim

        self.lstm = nn.LSTM(
            input_size=self.embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes) 

    def forward(self, input_ids):
        
        embedded = self.embedding(input_ids) 

        lstm_output, (hidden, cell) = self.lstm(embedded)
        hidden_forward = hidden[-2, :, :]  
        hidden_backward = hidden[-1, :, :]  
        hidden_concat = torch.cat((hidden_forward, hidden_backward), dim=1)

        hidden_concat = self.dropout(hidden_concat)
        logits = self.classifier(hidden_concat)

        return logits

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
def stance_type_bilstm(data, tokenizer, dataset_type,
                         embedding_dim=300, hidden_size=256, num_layers=2,
                         batch_size=16, num_epochs=10, learning_rate=2e-5):
    label2id = {'2': 0, '0': 1, '1': 2, '-1':3}
    cache_filename = f"embeddings_{dataset_type}_bilstm.npz"

    print(f"Подготовка данных для {dataset_type}...")
    texts = [item['text'] for item in data]
    labels = [label2id[item['stance']] for item in data]
    dataset = TextDataset(texts, labels, tokenizer)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
    all_true_labels = []
    all_pred_labels = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используется устройство: {device}")

    # Кросс-валидация
    for fold, (train_idx, val_idx) in enumerate(cv.split(texts, labels), 1):
        print(f"\n=== Fold {fold}/5 ===")
        train_dataset = torch.utils.data.Subset(dataset, train_idx)
        val_dataset = torch.utils.data.Subset(dataset, val_idx)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        vocab_size = tokenizer.vocab_size
        num_classes = 4 

        model = BiLSTMClassifier(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            num_classes=num_classes,
            dropout=0.5
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(num_epochs):
            model.train()
            total_loss = 0

            for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
                input_ids = batch['input_ids'].to(device)
                labels_batch = batch['label'].to(device)

                optimizer.zero_grad()
                outputs = model(input_ids)
                loss = criterion(outputs, labels_batch)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)

            model.eval()
            val_preds = []
            val_labels = []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(device)
                    labels_batch = batch['label'].to(device)

                    outputs = model(input_ids)
                    preds = torch.argmax(outputs, dim=1)

                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(labels_batch.cpu().numpy())

            val_acc = accuracy_score(val_labels, val_preds)
            print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}, Val Accuracy = {val_acc:.4f}')

        all_true_labels.extend(val_labels)
        all_pred_labels.extend(val_preds)

    # Вывод результатов
    print("\n=== Результаты кросс-валидации ===")
    print(classification_report(all_true_labels, all_pred_labels,
                                target_names=['for', 'against', 'other', 'no_position']))
    print(f"Accuracy: {accuracy_score(all_true_labels, all_pred_labels):.3f}")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")



In [ ]:
stance_type_bilstm(
    bilingualism_data,
    tokenizer,
    "biling_stance",
    embedding_dim=300,    
    hidden_size=256,        
    num_layers=2,           
    batch_size=16,          
    num_epochs=10,          
    learning_rate=2e-5     
)

In [ ]:
stance_type_bilstm(
    relativity_data,
    tokenizer,
    "stance",
    embedding_dim=300,    
    hidden_size=256,        
    num_layers=2,           
    batch_size=16,          
    num_epochs=10,          
    learning_rate=2e-5     
)

In [ ]:
stance_type_bilstm(
    ug_data,
    tokenizer,
    "ug_stance",
    embedding_dim=300,    
    hidden_size=256,        
    num_layers=2,           
    batch_size=16,          
    num_epochs=10,          
    learning_rate=2e-5     
)

Предобученная BiLSTM (из Lauscher et al. 2018)

In [ ]:
results = prepare_data_and_train_classifier(
        data=bilingualism_data,
        model_type='aspect',
        test_size=0.2,
        classifier_type='rf' 
    )


In [ ]:
results = prepare_data_and_train_classifier(
        data=relativity_data,
        model_type='aspect',
        test_size=0.2,
        classifier_type='rf' 
    )


In [ ]:
results = prepare_data_and_train_classifier(
        data=ug_data,
        model_type='aspect',
        test_size=0.2,
        classifier_type='rf' 
    )


RoBERTa large

In [ ]:
def stance(data, model, tokenizer, classifier, dataset_type):
    # мэппинг меток
    label2id = {'-1': 0, '0': 1, '1': 2, '2':3}

    # создаем имя файла
    cache_filename = f"stance_embeddings_{dataset_type}_{model.__class__.__name__}.npz"

    # эмбеддинги
    def get_cls_embedding(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        return cls_embedding

    # проверяем, есть ли сохраненные эмбеддинги
    import os
    if os.path.exists(cache_filename):
        try:
            # загружаем сохраненные эмбеддинги
            with np.load(cache_filename, allow_pickle=True) as saved_data:
                X = saved_data['X']
                y = saved_data['y']
            print(f"Загружены сохраненные эмбеддинги из {cache_filename}")
        except Exception as e:
            print(f"Ошибка при загрузке {cache_filename}: {e}")
            print("Вычисляются эмбеддинги заново...")
            X = []
            y = []

            for item in data:
                emb = get_cls_embedding(item['text'])
                X.append(emb)
                y.append(label2id[item['stance']])

            X = np.array(X)
            y = np.array(y)

            # сохраняем эмбеддинги
            np.savez(cache_filename, X=X, y=y)
            print(f"Эмбеддинги сохранены в {cache_filename}")
    else:
        # если файла нет, вычисляем эмбеддинги заново
        print(f"Файл {cache_filename} не найден. Вычисляются новые эмбеддинги для {dataset_type}...")
        X = []
        y = []

        for item in data:
            emb = get_cls_embedding(item['text'])
            X.append(emb)
            y.append(label2id[item['stance']])

        X = np.array(X)
        y = np.array(y)

        # сохраняем эмбеддинги
        np.savez(cache_filename, X=X, y=y)
        print(f"Эмбеддинги сохранены в {cache_filename}")

    if classifier == "LogisticRegression":
        clf = LogisticRegression(max_iter=1000)
    elif classifier == "LinearSVC":
        clf = LinearSVC(max_iter=5000)
    elif classifier == "GaussianNB":
        clf = GaussianNB()
    elif classifier == "RandomForest":
        clf = RandomForestClassifier(n_estimators=100, random_state=13)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
    # кросс-валидация
    y_pred = cross_val_predict(clf, X, y, cv=cv)

    print(classification_report(y, y_pred, target_names=['нерелевантно', 'против', 'нейтрально', 'за']))
    print(f"Accuracy: {accuracy_score(y, y_pred):.3f}")

    print("Метрики без учета нерелевантного класса")
    mask = y != 0

    print(classification_report(
        y[mask],
        y_pred[mask],
        labels=[1, 2, 3],
        target_names=['против', 'нейтрально', 'за']
    ))

    print(f"Accuracy: {accuracy_score(y[mask], y_pred[mask]):.3f}")

In [ ]:
stance(bilingualism_data, roberta_model, roberta_tokenizer, "LogisticRegression", "biling_stance")

In [ ]:
stance(ug_data, roberta_model, roberta_tokenizer, "LogisticRegression", "ug_stance")

In [ ]:
stance(relativity_data, roberta_model, roberta_tokenizer, "LogisticRegression", "relativity_stance")

SciBERT

In [ ]:
stance(bilingualism_data, scibert_model, scibert_tokenizer, "LogisticRegression", 'biling_stance')

In [ ]:
stance(ug_data, scibert_model, scibert_tokenizer, "LogisticRegression", 'ug_stance')

In [ ]:
stance(relativity_data, scibert_model, scibert_tokenizer, "LogisticRegression", 'relativity_stance')

Тестирование других классификаторов с лучшими вариантами моделей (SciBERT с контекстом для позиции)

In [ ]:
#LinearSVC
stance(bilingualism_data, scibert_model, scibert_tokenizer, "LinearSVC", "biling_stance")
stance(ug_data, scibert_model, scibert_tokenizer, "LinearSVC", "ug_stance")
stance(relativity_data, scibert_model, scibert_tokenizer, "LinearSVC", "relativity_stance")

In [ ]:
#GaussianNB
stance(bilingualism_data, scibert_model, scibert_tokenizer, "GaussianNB", "biling_stance")
stance(ug_data, scibert_model, scibert_tokenizer, "GaussianNB", "ug_stance")
stance(relativity_data, scibert_model, scibert_tokenizer, "GaussianNB", "relativity_stance")

In [ ]:
#RandomForestClassifier
stance(bilingualism_data, scibert_model, scibert_tokenizer, "RandomForest", "biling_stance")
stance(ug_data, scibert_model, scibert_tokenizer, "RandomForest", "ug_stance")
stance(relativity_data, scibert_model, scibert_tokenizer, "RandomForest", "relativity_stance")

LLM:zero-shot

In [ ]:
if __name__ == "__main__":
    main("universal grammar")

In [ ]:
if __name__ == "__main__":
    main("bilingualism")

In [ ]:
if __name__ == "__main__":
    main("linguistic relativity ")

LLM:few-shot

In [ ]:
if __name__ == "__main__":
    main_few_shot("bilingualism", task_type='stance')

In [ ]:
if __name__ == "__main__":
    main_few_shot("universal grammar", task_type='stance')

In [ ]:
if __name__ == "__main__":
    main_few_shot("linguistic relativity ", task_type='stance')